In [3]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_csv('../data/processed/telco_features.csv')

X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

THRESHOLD = 0.3

## Optuna search

In [4]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import recall_score

def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
            'random_state': 42,
            'n_jobs': -1,
            'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
            'eval_metric': 'logloss'
        }
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        y_pred = (proba >= THRESHOLD).astype(int)
        return recall_score(y_test, y_pred, pos_label=1)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print('Best Params', study.best_params)
print('Best Recall', study.best_value)

[I 2026-06-03 17:19:55,349] A new study created in memory with name: no-name-41fd96c1-cc9b-4b22-9eab-73d68164a05a
[I 2026-06-03 17:19:55,706] Trial 0 finished with value: 0.8689839572192514 and parameters: {'n_estimators': 614, 'learning_rate': 0.1396380498341774, 'max_depth': 5, 'subsample': 0.8172996374755293, 'colsample_bytree': 0.9449767473275477, 'min_child_weight': 7, 'gamma': 2.3271519217360206, 'reg_alpha': 0.6347468968568842, 'reg_lambda': 3.5761148216205294}. Best is trial 0 with value: 0.8689839572192514.
[I 2026-06-03 17:19:56,349] Trial 1 finished with value: 0.8155080213903744 and parameters: {'n_estimators': 461, 'learning_rate': 0.09683349700612301, 'max_depth': 8, 'subsample': 0.885308329248709, 'colsample_bytree': 0.7067114604982281, 'min_child_weight': 4, 'gamma': 0.20526024306798063, 'reg_alpha': 4.102045130222512, 'reg_lambda': 2.568906887387947}. Best is trial 0 with value: 0.8689839572192514.
[I 2026-06-03 17:19:56,691] Trial 2 finished with value: 0.863636363636

Best Params {'n_estimators': 419, 'learning_rate': 0.02920099108149599, 'max_depth': 6, 'subsample': 0.9549477232917243, 'colsample_bytree': 0.9952202461702226, 'min_child_weight': 10, 'gamma': 4.989055500009303, 'reg_alpha': 2.965629231014789, 'reg_lambda': 3.922303328892859}
Best Recall 0.9251336898395722


In [6]:
# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Add the scale_pos_weight and fixed params to the best ones from Optuna
best_params = study.best_params
best_params.update({
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "logloss"
})

# Create model from best params
xgb = XGBClassifier(**best_params)

# Training timer
start_train = time.time()
xgb.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"training time: {train_time:.2f} seconds")

# Prediction timer
start_pred = time.time()
proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)
pred_time = time.time() - start_pred
print(f"prediction time: {pred_time:.4f} seconds")

# Classification report
print(classification_report(y_test, y_pred, digits=3))

training time: 0.27 seconds
prediction time: 0.0019 seconds
              precision    recall  f1-score   support

           0      0.954     0.563     0.708      1033
           1      0.434     0.925     0.591       374

    accuracy                          0.660      1407
   macro avg      0.694     0.744     0.650      1407
weighted avg      0.816     0.660     0.677      1407



the tuned model hits very high recall (~0.9+) but precision drops (~0.45) — it flags many loyal customers. 